# Konu 10 — Video üretimi

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 9'da bir görsel ızgarası ürettik; her görsel birkaç saniyede geldi. Video öyle değil: isteği gönderirsin,
sonuç **hemen gelmez**; hazır olması birkaç dakika sürebilir. Bugünün yeni fikri bu: **uzun süren iş.** İsteği başlat, sana verilen
**iş numarasını** sakla, arada bir "bitti mi?" diye sor, bitince indir.

Videolar hocanın hesabından, bir **aracı hizmet** üzerinden üretiliyor. Hocanın anahtarını görmüyorsun; sana
özel bir **video kodun** var. Her klibin bir fiyatı var ve hoca ödüyor; aracı hizmet öğrenci başına klip sayısını sınırlıyor.

Isınma, Adım 1 ve Adım 2 anahtarsız; Isınma ve Adım 2 internetsiz de çalışır. Adım 3'ten sonrası `anahtar.txt`'deki iki yeni satırı
(`VIDEO_ADRES`, `VIDEO_KOD`) ister. Defter 2 klip üretir (her biri 4 saniye); Bonus ile 3.

## Isınma — Bir video kaç para?

Görseller ücretsiz kotadan geliyordu. Video saniye başına ücretlidir. Dersin modeli Gemini Omni Flash; Gemini API
fiyat sayfasına göre 720p'de (1280 × 720 piksel) saniyesi **yaklaşık 0,10 dolar** (Ekim 2026). Aracı hizmet çözünürlüğü 720p'ye
kilitliyor. Fiyatı **sabit** olarak yazıyoruz (Konu 9: büyük harf). Python ondalık sayıları virgülle değil **noktayla** yazar: `0.10`.

In [ ]:
UCRET_SANIYE = 0.10

sure = 4
print(sure, "saniyelik klip:", sure * UCRET_SANIYE, "USD")

Bir müşteri 20 saniyelik bir tanıtım videosu istiyor. Aracı hizmet klipleri 4, 6 ya da 8 saniye üretiyor; daha uzun
bir video, klipleri kurguda birleştirerek yapılır. 20 saniyeyi 8 + 8 + 4 ile kuruyoruz. Toplam süreyi Konu 1'deki gibi sayaçla buluyoruz:

In [ ]:
klipler = [8, 8, 4]
toplam = 0
for klip in klipler:
    toplam = toplam + klip

print("Toplam süre:", toplam, "saniye")
print("Ücret:", toplam * UCRET_SANIYE, "USD")

Müşterinin bütçesi bunun yarısı. Çözünürlük kilitli; oranı 16:9 ya da 9:16 seçmek fiyatı değiştirmiyor.

**Çalıştırmadan önce tahmin et, sohbete yaz:** aynı işi yarı fiyatına nasıl yaparsın? Kaç klip, toplam kaç saniye?

In [ ]:
kisa_klipler = [4, 6]
toplam_kisa = 0
for klip in kisa_klipler:
    toplam_kisa = toplam_kisa + klip

print("Toplam süre:", toplam_kisa, "saniye")
print("Ücret:", toplam_kisa * UCRET_SANIYE, "USD")

Fiyatı yalnızca **süre** belirliyor: yarı fiyat, yarı süre. 20 saniye yerine 10 saniyelik bir kurgu (iki klip). Ya da tek tek:
8 saniyelik klip yerine 4 saniyelik klip yarı fiyat. **Süre kararı bir bütçe kararıdır:** sosyal medyada izlenen bir tanıtım için
10 saniye yeter mi? Hangi planları atardın?

Son olarak video ile görseli karşılaştıralım. Konu 9'da bir görsel ~58 neuron tutuyordu; Cloudflare ücretsiz kotayı
aşanlardan 1.000 neuron için 0,011 dolar alıyor.

**Tahmin et:** 4 saniyelik bir klibin parasıyla kaç görsel üretilir? 10 mu, 100 mü, 1.000 mi?

In [ ]:
gorsel_ucreti = 58 * 0.011 / 1000
klip_ucreti = 4 * UCRET_SANIYE

print("Bir görsel:", gorsel_ucreti, "USD")
print("Bir klip kaç görsel eder:", klip_ucreti / gorsel_ucreti)

Yaklaşık 600. Video, art arda gelen onlarca kareden oluşur (saniyede 24 kare: 4 saniye yaklaşık 100 kare); karelerin birbirini tutması
ve ses de işin içinde. Bir video denemesi yüzlerce görsel denemesine bedel: önce görselle (ucuz) dene, beğendiğini videoya çevir.

## Adım 1 — Arayüzden bir klip (kod yok)

Konu 7'deki gibi önce arayüzden. Öğrenci hesabınla (Google AI Plus öğrenci teklifi) Gemini uygulamasında video üretme
seçeneğini aç ve kısa bir yönerge yaz, ör. "masada buharı tüten bir seramik fincan, kamera yavaşça yaklaşıyor".
Hesabın yoksa hocanın ekranından izle.

**Sohbete yaz:** sonuç kaç saniyede geldi? Beklerken ekranda ne gördün (bir yükleme çubuğu, bir mesaj, hiçbir şey)?

Arayüz beklemeyi senin yerine yönetiyor: isteği gönderiyor, arka planda soruyor, bitince gösteriyor.
Kodda bunu **sen** yapacaksın.

## Adım 2 — Uzun süren iş: önce sahte bir işle

Gerçek video beklemeden önce internetsiz, **sahte** bir işle deneyelim. `video.py` bu klasördeki yardımcı dosya; içindeki
iki fonksiyon gerçek aracı hizmet gibi cevap veriyor:

- `sahte_baslat(govde)`: işi başlatır, bir **iş numarası** verir.
- `sahte_durum(is_no)`: o işin durumunu söyler. Sahte iş **5 saniye** sürer; o zamana kadar `{'bitti': False}` der.

`govde` gerçek istekte de aynı olacak: yönerge (İngilizce, Konu 7), süre ve oran. Süre tırnak içinde: aracı hizmet onu
metin olarak bekliyor (Konu 8'deki form alanları gibi).

In [ ]:
import time
from video import sahte_baslat, sahte_durum

govde = {
    "yonerge": "steam rising from a ceramic coffee cup on a wooden desk, soft morning light, slow camera push in",
    "sure": "4",
    "oran": "16:9",
}
yanit = sahte_baslat(govde)
print(yanit)

Video gelmedi; yalnızca bir iş numarası geldi. Gerçek hizmette işin kendisi sunucuda sürer. Numarayı bir değişkende saklıyoruz ve soruyoruz:

In [ ]:
is_no = yanit["is_no"]
durum = sahte_durum(is_no)
print(durum)

`{'bitti': False}`: henüz bitmedi. Bu hücreyi birkaç saniye arayla birkaç kez daha çalıştır (**Ctrl + Enter** hücrede kalır).
5 saniye geçince `{'bitti': True, 'video': ...}` görürsün.

Elle sormak yorucu. Döngüye verelim: en fazla 10 kez sor, her sorudan sonra 2 saniye bekle (`time.sleep`, Konu 9).
Bittiyse `break` ile döngüden çık. Hücre yeni bir sahte iş başlatıyor ki saat sıfırdan başlasın.

- `range(10)`: "10 kez". `deneme` 0'dan başlar; ekrana `deneme + 1` yazıyoruz ki sorular 1'den saysın.
- Döngü 10 soruda bitmezse aşağıdaki `if` "zaman aşımı" der (`if not durum["bitti"]`: "iş bitmediyse"; `not` koşulu tersine çevirir). Bu, sonsuza kadar beklememenin yolu: **zaman aşımı.**

**Çalıştırmadan önce tahmin et, sohbete yaz:** iş 5 saniye sürüyor, döngü 2 saniyede bir soruyor. Kaç kez `Henüz bitmedi` yazar?

In [ ]:
yanit = sahte_baslat(govde)
is_no = yanit["is_no"]

for deneme in range(10):
    durum = sahte_durum(is_no)
    if durum["bitti"]:
        print("Hazır:", durum["video"])
        break
    print("Henüz bitmedi, soru", deneme + 1)
    time.sleep(2)

if not durum["bitti"]:
    print("Zaman aşımı. İş numarası:", is_no)

Üç kez: 0., 2. ve 4. saniyede iş sürüyordu, 6. saniyede bitmişti. Bu döngü en fazla 10 × 2 = 20 saniye bekler.

**Dene:** `range(10)` yerine `range(2)` yazıp hücreyi çalıştır: zaman aşımı. Sonra yukarıdaki tek soruluk hücreyi
(`sahte_durum(is_no)`) çalıştır: iş **bitmiş**. Zaman aşımı işi iptal etmez; yalnızca senin beklemeni bitirir.
İş numaran elindeyse sonra yeniden sorarsın. (Denedikten sonra `range(10)`'a geri döndür.)

"Çalışmadı" sandığımız şeylerin çoğu "henüz bitmedi"dir. Bu yüzden iyi bir program ne olduğunu söyler: "bekliyorum, şu kadar
saniye oldu." Arayüzlerdeki yükleme göstergesi de aynı işi yapar.

## Adım 3 — Video kodunu oku

`anahtar.txt`'ye derste iki satır ekledik: aracı hizmetin adresi ve sana özel video kodun.

```
VIDEO_ADRES = https://...
VIDEO_KOD = ...
```

Video kodu senin kimliğin: aracı hizmet klip sayını onunla tutar. Kimseyle paylaşma, ödevle birlikte gönderme.
`anahtarlari_oku` (`video.py`) Konu 9'daki `anahtari_oku`'nun aynısı; bu kez bütün satırları bir sözlük olarak veriyor.

In [ ]:
import requests
from pathlib import Path
from IPython.display import Video
from video import anahtarlari_oku

Video kodu başlıkta gidiyor, Konu 2'deki gibi: `Bearer` ve bir boşluk, sonra kod.

In [ ]:
anahtarlar = anahtarlari_oku()
VIDEO_ADRES = anahtarlar["VIDEO_ADRES"]
VIDEO_KOD = anahtarlar["VIDEO_KOD"]
basliklar = {"Authorization": "Bearer " + VIDEO_KOD}

print("Aracı hizmet:", VIDEO_ADRES)

## Adım 4 — Gerçek işi başlat

Adım 2'deki `govde`'yi bu kez aracı hizmete gönderiyoruz. Konu 2'deki `requests.post`; adres aracı hizmetin adresi ve `/baslat`.
`timeout=60`: cevap 60 saniyede gelmezse istek bırakılır (Konu 9'daki yardımcı dosyada da vardı).

**Bu hücreyi bir kez çalıştır.** Her çalıştırma yeni bir iş başlatır: yeni bir klip, yeni bir ücret, klip hakkından bir eksik.

In [ ]:
adres = VIDEO_ADRES + "/baslat"
cevap = requests.post(adres, headers=basliklar, json=govde, timeout=60)
print(cevap.status_code)
yanit = cevap.json()
print(yanit)

`200` ve bir iş numarası geldiyse devam. `200` değilse yanıtta bir `hata` metni var (ör. video kodu yanlış, klip hakkın doldu);
aşağıdaki hücreyi çalıştırma, hata metnini oku.

In [ ]:
is_no = yanit["is_no"]
print("İş numaram:", is_no)

İş numaranı bir kenara yaz. Defter kapanırsa ya da bağlantı koparsa iş sunucuda sürmeye devam eder; numarayla yeniden sorarsın
ve yeni bir ücret ödemezsin.

## Adım 5 — Bekle: aralıklarla sor

Adım 2'deki döngünün gerçek hâli. Bu kez durumu `requests.get` ile soruyoruz: **get** "bana şunu ver" demek, gövde göndermez;
`post` ise gövdeyle bir şey gönderir (Konu 2). Durum adresi `/durum/` ve iş numarası.

Döngüyü bir fonksiyona koyuyoruz, çünkü Adım 7'de bir kez daha bekleyeceğiz:

- Cevapta `hata` varsa iş başarısız oldu (ör. model yönergeyi reddetti): yaz ve `None` döndür.
- `bitti` ise videonun adresini döndür. `return` fonksiyonu orada bitirir (Konu 9).
- Değilse ne kadar beklediğimizi yaz, 10 saniye bekle, yeniden sor.
- En fazla 36 kez: 36 × 10 saniye = **6 dakika**. Bir video birkaç dakika sürebilir; 6 dakikayı sınır seçtik. Döngü
  biterse zaman aşımı; iş numarasını yazıp `None` döndürür.
- `deneme * 10` yaklaşık geçen süre: her sorudan sonra 10 saniye bekliyoruz.
- `timeout=30` tek bir sorunun sınırı; döngünün sınırı ise bütün bekleyişin sınırı. İkisi farklı şeyler.

In [ ]:
def bekle(is_no):
    durum_adresi = VIDEO_ADRES + "/durum/" + is_no
    for deneme in range(36):
        cevap = requests.get(durum_adresi, headers=basliklar, timeout=30)
        durum = cevap.json()
        if "hata" in durum:
            print("Hata:", durum["hata"])
            return None
        if durum["bitti"]:
            print("Hazır, yaklaşık", deneme * 10, "saniyede")
            return durum["video"]
        print("Henüz bitmedi, geçen süre:", deneme * 10, "saniye")
        time.sleep(10)
    print("Zaman aşımı. İş sürüyor olabilir; iş numarası:", is_no)
    return None

Bu hücre birkaç dakika sürebilir. Beklerken sohbette: Adım 1'deki arayüz kaç saniye sürmüştü?

In [ ]:
video_adresi = bekle(is_no)
print(video_adresi)

`None` yazdıysa üstteki mesaja bak: `Hata: ...` ise iş başarısız; `Zaman aşımı` ise iş hâlâ sürüyor olabilir, bu hücreyi
yeniden çalıştır (yeni ücret yok). Bağlantı hatasıyla hücre durursa yine aynı hücreyi çalıştır.

## Adım 6 — İndir ve izle

Videolar için bir klasör (Konu 8):

In [ ]:
klasor = Path("videolar")
klasor.mkdir(exist_ok=True)
metin_yolu = klasor / "metinden.mp4"

Video bu kez **base64 değil, doğrudan bayt** olarak geliyor: `cevap.content` (Konu 8'deki araçta `sdxl` de böyleydi).
Baytları Konu 8'deki gibi `"wb"` ile yazıyoruz. Üstteki hücre `None` yazdıysa bunu çalıştırma.

In [ ]:
cevap = requests.get(video_adresi, headers=basliklar, timeout=120)
print(cevap.status_code)
with open(metin_yolu, "wb") as dosya:
    dosya.write(cevap.content)

print("Kaydedildi:", metin_yolu)

`Video` videoyu defterin içinde oynatır. `embed=True`: videoyu defterin içine göm; VS Code başka türlü göstermiyor.

In [ ]:
Video(metin_yolu, embed=True)

Tasarımcı gözüyle izle ve sohbete yaz:

- Yönergede "slow camera push in" (kamera yavaşça yaklaşır) yazdık. Kamera bunu yaptı mı?
- Buhar nasıl hareket ediyor? Fizik inandırıcı mı; nerede "yapay" duruyor?
- Sesi aç: model ses de üretiyor. Yönergede ses yoktu; ne duyuyorsun? Sen ne isterdin?

Hareket yönergesi bir **kamera dili**: yaklaşır (push in), uzaklaşır (pull out), yana kayar (pan), yukarı kalkar (tilt up).
Konu 7'deki yönerge anatomisine yeni bir katman.

## Adım 7 — Görselden video

Konu 9'da seçtiğin görseli videonun **ilk karesi** yapıyoruz. Aşağıdaki yolda kendi kazananının adını yaz (Konu 9 Adım 6'daki
`benim_secimim`; ödevin görselleri `odev9/` içinde). Defter `10-video` klasöründe çalışıyor; `..` bir üst klasör.
Görsel yoksa (Konu 9'u çalıştırmadıysan) defter `veri/` klasöründeki yedek görseli kullanır.

In [ ]:
gorsel_yolu = Path("../09-toplu-uretim/izgara/suluboya_adacayi.jpg")
if not gorsel_yolu.exists():
    print("Konu 9'un görseli yok; yedek görseli kullanıyorum")
    gorsel_yolu = Path("veri/yedek_fincan.jpg")

print(gorsel_yolu)

Görseli gövdeye koymak için metne çevirmemiz gerekiyor: JSON bayt taşıyamaz. Konu 8'de base64 **metnini** baytlara çözmüştük
(`b64decode`); şimdi tersi: baytları metne (`b64encode`). `b64encode` sonucu yine bayt olarak verir (başında `b'`); `.decode()`
onu düz metne çevirir.

In [ ]:
import base64

with open(gorsel_yolu, "rb") as dosya:
    baytlar = dosya.read()
kodlu = base64.b64encode(baytlar)
metin = kodlu.decode()

uzunluk = len(metin)
print(uzunluk, "karakter")

Gövdeye `"gorsel"` alanı ekleniyor. Yönerge artık ne olduğunu değil, **ne olacağını** anlatıyor: sahne görselde zaten var.
Bu hücreyi de bir kez çalıştır.

**Çalıştırmadan önce tahmin et, sohbete yaz:** görselin kare (1:1), video yatay (16:9). Model kenarları ne yapacak?

In [ ]:
gorsel_govde = {
    "yonerge": "slow camera push in, gentle natural motion, soft morning light",
    "sure": "4",
    "oran": "16:9",
    "gorsel": metin,
}
adres = VIDEO_ADRES + "/baslat"
cevap = requests.post(adres, headers=basliklar, json=gorsel_govde, timeout=60)
print(cevap.status_code)
yanit = cevap.json()
print(yanit)

Yine `200` değilse hata metnini oku (ör. görselde bir insan yüzü varsa reddedilebilir). Aynı `bekle` fonksiyonu:

In [ ]:
gorsel_is_no = yanit["is_no"]
video_adresi = bekle(gorsel_is_no)

In [ ]:
gorsel_video_yolu = klasor / "gorselden.mp4"
cevap = requests.get(video_adresi, headers=basliklar, timeout=120)
print(cevap.status_code)
with open(gorsel_video_yolu, "wb") as dosya:
    dosya.write(cevap.content)

Video(gorsel_video_yolu, embed=True)

İlk kareyi Konu 9'daki görselle karşılaştır ve sohbete yaz:

- Kompozisyon korundu mu? Stil (suluboya, linol...) hareket boyunca korundu mu, yoksa video "fotoğrafa" mı kaydı?
- Kare görsel yatay videoya nasıl sığdı: kırpıldı mı, kenarlar uyduruldu mu?
- Hangi görseller harekete uygun? Derinliği, boşluğu, yön veren çizgileri olanlar mı; düz bir afiş mi?

## Adım 8 — Üretim günlüğü

Konu 9'daki gibi her videonun nasıl üretildiğini kaydediyoruz; bu kez **maliyet** de kayıtta. Her video bir sözlük (bir kart):

In [ ]:
metin_kaydi = {
    "dosya": str(metin_yolu),
    "gorsel": "yok",
    "yonerge": govde["yonerge"],
    "model": "gemini-omni-1.1-flash",
    "sure": govde["sure"],
    "oran": govde["oran"],
    "cozunurluk": "720p",
    "is_no": is_no,
    "maliyet_usd": 4 * UCRET_SANIYE,
}
gorsel_kaydi = {
    "dosya": str(gorsel_video_yolu),
    "gorsel": str(gorsel_yolu),
    "yonerge": gorsel_govde["yonerge"],
    "model": "gemini-omni-1.1-flash",
    "sure": gorsel_govde["sure"],
    "oran": gorsel_govde["oran"],
    "cozunurluk": "720p",
    "is_no": gorsel_is_no,
    "maliyet_usd": 4 * UCRET_SANIYE,
}
kayitlar = [metin_kaydi, gorsel_kaydi]

`str(...)`: yol bir `Path`; JSON onu yazamaz, metne çeviriyoruz. Gövdeyi olduğu gibi kaydetmiyoruz: içinde görselin on binlerce
karakterlik base64 metni var. Toplam maliyeti sayaçla bulup günlüğü yazıyoruz:

In [ ]:
import json

toplam_ucret = 0
for kayit in kayitlar:
    toplam_ucret = toplam_ucret + kayit["maliyet_usd"]
print("Bu defterin videoları:", toplam_ucret, "USD")

with open("video_gunlugu.json", "w", encoding="utf-8") as dosya:
    json.dump(kayitlar, dosya, ensure_ascii=False, indent=2)
print("Günlük yazıldı")

Sınıfın tamamı bu defteri çalıştırınca 12 öğrenci × 0,80 = yaklaşık 10 dolar. Görselde aynı para 15.000 görsel ederdi.
Günlükteki maliyet alanı final projesinde de olacak: kampanyanın kaç dolara mal olduğunu bilmek tasarımcının işi.

## Bonus

**1. Dikey video (bir klip daha; klip hakkın varsa).** Aynı yönerge, 9:16 (telefon, hikâye). Kompozisyon yeniden nasıl kuruldu?
Fincan nerede duruyor, ne kırpıldı? Fiyat değişti mi?

In [ ]:
dikey_govde = {
    "yonerge": govde["yonerge"],
    "sure": "4",
    "oran": "9:16",
}
cevap = requests.post(adres, headers=basliklar, json=dikey_govde, timeout=60)
yanit = cevap.json()
print(yanit)

In [ ]:
dikey_is_no = yanit["is_no"]
video_adresi = bekle(dikey_is_no)

dikey_yolu = klasor / "dikey.mp4"
cevap = requests.get(video_adresi, headers=basliklar, timeout=120)
with open(dikey_yolu, "wb") as dosya:
    dosya.write(cevap.content)

Video(dikey_yolu, embed=True)

**2. İş numarasıyla yeniden sor (ücretsiz).** Adım 4'teki işin numarası `is_no`'da duruyor. Onu bir kez daha soralım.
**Tahmin et:** kaç kez `Henüz bitmedi` yazar?

In [ ]:
tekrar = bekle(is_no)
print(tekrar)

Hiç: iş çoktan bitti, cevap ilk soruda gelir. Yeni bir iş başlamadı, yeni bir ücret yok. Konu 9'daki "kaldığı yerden devam"ın
video hâli: pahalı iş bir kez yapılır, sonucu numarasıyla geri alınır.

**3. Yardımcı dosyayı oku.** `video.py`'yi VS Code'da aç. `video_uret` bugün yazdığın her şeyi tek fonksiyonda yapıyor:

| `video_uret`'in içinde | Bu defterde |
|---|---|
| `yol` zaten varsa hiç istek göndermiyor | Konu 9'un "zaten var mı?" sorusu (video pahalı) |
| `govde`, isteğe bağlı `"gorsel"` (base64) | Adım 2, Adım 7 |
| `requests.post(... "/baslat" ...)` | Adım 4 |
| `for deneme in range(en_fazla)` ve `requests.get(... "/durum/" ...)` | Adım 5'teki `bekle` |
| `try` / `except`: bağlantı koparsa beklemeyi bırakmıyor, biraz sonra yeniden soruyor | Konu 9'daki `sabirli_ciz` |
| `yol.write_bytes(cevap.content)` | Adım 6 (`open(..., "wb")` + `write`) |

Konu 11'de üretim hattının son (isteğe bağlı) halkası bu fonksiyon olacak.